# ReBRAC Stage D Phase 2 — `worldcomp-1000` privileged-critic formal

**目的**：Phase 1 deployable formal 拿到 `mean=0.928 ± 0.077`（落入情景 A），并暴露两个新机制点：(1) `β2·critic_penalty_ratio = 0.0111`，dual penalty 在 worldcomp 上几乎退化为单 penalty；(2) seed 44 在 deployable 上掉到 `0.78`，是当前 Stage D 最有信息量的诊断点。

按 [plan §6.5.3](../docs/rebrac_experiment_plan.md) 收紧后的设计：情景 A 下 Phase 2 是 **3-seed 边际确认**，**必须包含 seed 44**（不是任意 3 个 seed），用 privileged-critic 轨道直接区分 seed 44 是 critic 信息瓶颈还是数据/优化层面 outlier。

**核心问题**：

1. ReBRAC privileged-critic 是否进一步把 deployable→teacher 的 gap 关闭？要求 `mean_test_success > 0.922`（TD3BC privileged-critic formal）且 `std ≤ 0.10`。
2. seed 44 在 deployable 上的 `0.78` 离群点，是否被 privileged critic 救回？
   - 如果救回 → critic 侧的 OOD 风险 / 信息不足是 seed 44 难训的真因；
   - 如果救不回 → seed 44 与 critic 无关，是数据/优化层面的 outlier；需要回到 collector 检查 seed 44 触发的 episode 起点分布。

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 / β2 | `4.0 / 2.0` | Stage C 锁定的 finalist；Stage D 不扫超参 |
| dataset | `worldcomp-1000` | 与 Phase 1 同 dataset |
| seeds | `42 43 44` | **3-seed 边际确认，必须含 seed 44**；seeds 42/43 与 Phase 1 / probe 对齐 |
| TRAIN_EPOCHS | `64` | 与 Phase 1 一致；epoch-probe 决定 |
| 轨道 | **privileged-critic** | actor 用 deployable obs；critic 用 privileged obs；actor update mode `zeros`（与 TD3BC worldcomp phase0c 完全对齐） |
| val manifest | 40 episodes | 与 Phase 1 对齐 |
| test manifest | 100 episodes | 与 Phase 1 / Stage C / TD3BC worldcomp formal 同口径 |

**预算**：3 seeds × 1 finalist × 64 epoch ≈ Phase 1 deployable 的 60% 算力。manifest 复用 Phase 1 已生成的 `benchmarks/offline_rebrac_worldcomp_final/`。

**输出树**（与 Phase 1 deployable 完全隔离）：
- `checkpoints/offline/rebrac/worldcomp_teacher_gap/privileged_critic/`
- `results/offline/rebrac/worldcomp_teacher_gap/privileged_critic/`

**Phase 2 通过判据**（详见 [plan §6.5.3](../docs/rebrac_experiment_plan.md)）：

| 条件 | 阈值 | 说明 |
| --- | --- | --- |
| 均值 | `mean_test_success > 0.922`（TD3BC privileged-critic formal） | "ReBRAC privileged 进一步追平/超过 TD3BC privileged" |
| 方差 | `std ≤ 0.10` | 与 Phase 1 / Stage C 一致 |
| gap closure（情景 A） | `> 50%`（与 TD3BC 的 48.5% 比较） | 注意 Phase 1 deployable 已经达到 53%，因此 Phase 2 真正回答的是 "privileged critic 是否在 ReBRAC 上仍是有意义的 gap 诊断工具" |
| seed 44 诊断 | `test_succ ≥ 0.85`（与 Stage C crosscomp seed 44 量级对齐） | 区分 critic 瓶颈 vs 数据/优化 outlier |

## 0. 环境 sanity check

In [1]:
!lscpu | head -10
print()
!nvidia-smi

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  12
On-line CPU(s) list:                     0-11
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   85

Wed Apr 29 13:50:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-U

In [2]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 1. 环境配置

In [4]:
import os

# —— 通用（与其它 ReBRAC / TD3BC 实验一致）——
os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"

# —— Stage C 锁定的 finalist（Stage D 不扫超参）——
os.environ["ACTOR_PENALTY_COEF"]  = "4.0"
os.environ["CRITIC_PENALTY_COEF"] = "2.0"

# —— Phase 2 privileged-critic 协议（覆盖 driver 默认）——
# 必须含 seed 44，不是任意 3 seed（理由见 plan §6.5.3）
os.environ["PRIVILEGED_FINAL_SEEDS"]                       = "42 43 44"
os.environ["PRIVILEGED_FINAL_TRAIN_EPOCHS"]                = "64"   # 与 Phase 1 一致
os.environ["PRIVILEGED_FINAL_CHECKPOINT_EVERY_EPOCHS"]     = "8"
os.environ["PRIVILEGED_ACTOR_UPDATE_MODE"]                 = "zeros"  # 与 TD3BC worldcomp phase0c 对齐
os.environ["FINAL_VAL_MANIFEST_EPISODES"]                  = "40"
os.environ["FINAL_TEST_MANIFEST_EPISODES"]                 = "100"

# 其余（DATASET_POLICY=worldcomp, DATASET_EPISODES_VALUE=1000,
# BENCHMARK_KEY=single_u10_cross_tgt15, PROBE_LAYOUT=s0,
# HISTORY_LENGTH=4, TASK_GEOMETRY=cross_stream, TARGET_SPEED=1.5,
# OBJECTIVE=efficiency_v2, SAMPLING_MODE=shuffle_no_replacement,
# BATCH_SIZE=256，FINAL_MANIFEST_ROOT=benchmarks/offline_rebrac_worldcomp_final）
# 使用 driver 默认值；manifest 由 Phase 1 已生成。

## 2. Manifest / 数据集复用 sanity check

driver 默认 `FINAL_MANIFEST_ROOT=benchmarks/offline_rebrac_worldcomp_final`，与 Phase 1 deployable 共用。`worldcomp-1000` 数据集应已在 epoch-probe / Phase 1 阶段 collect 过；这里只是显式 sanity check。

In [5]:
import pathlib

manifest_val  = pathlib.Path("benchmarks/offline_rebrac_worldcomp_final/val_40/single_u10_cross_tgt15.json")
manifest_test = pathlib.Path("benchmarks/offline_rebrac_worldcomp_final/test_100/single_u10_cross_tgt15.json")
dataset       = pathlib.Path("offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz")

for label, p in [("val manifest", manifest_val), ("test manifest", manifest_test), ("offline dataset", dataset)]:
    if p.exists():
        print(f"[reuse] {label} 已存在：{p}")
    else:
        print(f"[warn] {label} 缺失：{p}（driver 会自动重建/重收）")

[reuse] val manifest 已存在：benchmarks/offline_rebrac_worldcomp_final/val_40/single_u10_cross_tgt15.json
[reuse] test manifest 已存在：benchmarks/offline_rebrac_worldcomp_final/test_100/single_u10_cross_tgt15.json
[reuse] offline dataset 已存在：offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz


## 3. 全流程：privileged_train → validate（每个 ckpt）→ select → test → summarize

3 个 train run（3 seeds × 1 finalist × 64 epoch），每个产出 8 个 ckpt（每 8 epoch 一个），每个 ckpt 上跑 val=40。selection 按 `success_rate → return → -safety_cost → -time` 选最佳，最佳 ckpt 在 test=100 上重跑一次作为该 (seed) 的最终成绩。

driver 会自动：
- `USE_ASYMMETRIC_CRITIC=1`：critic 看到 `privileged_obs`（body-frame `[u_eq, v_eq]`）；
- `PRIVILEGED_ACTOR_UPDATE_MODE=zeros`：actor 改进步骤 `Q(s, π_θ(s))` 时把 `privileged_obs` 置零；
- 输出隔离到 `worldcomp_teacher_gap/privileged_critic/` 子目录。

In [ ]:
# 跑 Phase 2 privileged-critic 全流程
for mode in ["privileged_train", "privileged_validate", "privileged_test", "privileged_summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh


[cmd] env MODE=train PYTHON_BIN=python3 DEVICE=cuda BENCHMARK_KEY=single_u10_cross_tgt15 FLOW_PATH=wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy TASK_GEOMETRY=cross_stream TARGET_SPEED=1.5 OBJECTIVE=efficiency_v2 PROBE_LAYOUT=s0 HISTORY_LENGTH=4 DATASET_POLICY=worldcomp DATASET_EPISODES=1000 DATASET_SEED=0 COLLECT_WORKERS=8 ACTOR_PENALTY_COEFS=4.0 CRITIC_PENALTY_COEFS=2.0 BATCH_SIZE=256 DROP_LAST_BATCH=0 SAMPLING_MODE=shuffle_no_replacement HIDDEN_DIM=256 NUM_HIDDEN_LAYERS=3 ACTOR_LR=3e-4 CRITIC_LR=3e-4 GAMMA=0.99 TAU=0.005 POLICY_NOISE=0.2 NOISE_CLIP=0.5 POLICY_FREQ=2 GRAD_CLIP_NORM=10.0 NORMALIZER_EPS=1e-3 LOG_EVERY=1000 TRAIN_METRICS_WINDOW_FRACTION=0.25 EVAL_WORKERS=6 EVAL_WORKER_DEVICE=cpu VALIDATION_SEED=123 TEST_SEED=456 FORCE_REEVAL=0 SEEDS=42 43 44 TRAIN_EPOCHS=64 CHECKPOINT_EVERY_EPOCHS=8 VAL_MANIFEST_EPISODES=40 TEST_MANIFEST_EPISODES=100 MANIFEST_ROOT=benchmarks/offline_rebrac_worldcomp_final CHECKPOINT_ROOT=checkpoints/offline/rebrac/worldcomp_teacher_g

## 4. 主结果：3-seed test overview + per-seed 分布

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS_ROOT = Path("results/offline/rebrac/worldcomp_teacher_gap/privileged_critic")
DATASET_NAME = "worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
PAIR_TAG     = "actorb_4p0__criticb_2p0"
SEEDS        = os.environ["PRIVILEGED_FINAL_SEEDS"].split()


def load_test_per_seed() -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        path = RESULTS_ROOT / DATASET_NAME / PAIR_TAG / "test" / f"seed_{seed}.json"
        if not path.exists():
            print(f"[warn] missing test json: {path}")
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "seed": seed,
            "success_rate": payload["eval_success_rate"],
            "return": payload["eval_return"],
            "safety_cost": payload["eval_safety_cost"],
            "time_s": payload["eval_time_s"],
            "progress_ratio": payload.get("eval_progress_ratio"),
            "path_efficiency": payload.get("eval_path_efficiency"),
        })
    return pd.DataFrame(rows)


per_seed = load_test_per_seed()
print("[per-seed test results — privileged-critic]")
print(per_seed.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

if not per_seed.empty:
    print()
    print("[summary — privileged-critic]")
    print(f"  mean test success_rate = {per_seed['success_rate'].mean():.4f}")
    print(f"  std  test success_rate = {per_seed['success_rate'].std():.4f}")
    print(f"  mean test return       = {per_seed['return'].mean():.3f}")
    print(f"  std  test return       = {per_seed['return'].std():.3f}")
    print(f"  mean test safety_cost  = {per_seed['safety_cost'].mean():.3f}")

[per-seed test results — privileged-critic]
seed  success_rate  return  safety_cost  time_s  progress_ratio  path_efficiency
  42        0.9600 22.3022       7.2798 54.5240          0.8968           0.7665
  43        0.9200 -2.2102       9.9063 61.7270          0.7831           0.7245
  44        0.9000 15.8793       7.1983 54.0160          0.8907           0.7601

[summary — privileged-critic]
  mean test success_rate = 0.9267
  std  test success_rate = 0.0306
  mean test return       = 11.990
  std  test return       = 12.711
  mean test safety_cost  = 8.128


In [ ]:
# overview summary（critic_penalty / target_q）
import csv

overview_path = RESULTS_ROOT / "summaries" / "overview.csv"
if overview_path.exists():
    with overview_path.open(encoding="utf-8") as fp:
        reader = csv.DictReader(fp)
        for row in reader:
            print(f"[overview] dataset={row['dataset']} pair={row['pair']} num_seeds={row['num_seeds']}")
            print(f"  mean_test_success_rate = {float(row['mean_test_success_rate']):.4f}")
            print(f"  std_test_success_rate  = {float(row['std_test_success_rate']):.4f}")
            print(f"  mean_test_return       = {float(row['mean_test_return']):.3f}")
            print(f"  mean_critic_penalty    = {float(row['mean_critic_penalty']):.4f}")
            print(f"  mean_target_q          = {float(row['mean_target_q']):.3f}")
            penalty_ratio = float(row['mean_critic_penalty_ratio'])
            beta2 = float(os.environ['CRITIC_PENALTY_COEF'])
            print(f"  mean_critic_penalty_ratio = {penalty_ratio:.4f}  (β2 · ratio = {beta2 * penalty_ratio:.4f})")
            print()
            print("  对照（Phase 1 deployable 同 finalist）:")
            print(f"    mean_test_success_rate = 0.9280  (Δ priv − dep = {float(row['mean_test_success_rate']) - 0.9280:+.4f})")
            print(f"    mean_target_q           = 15.220 (Δ priv − dep = {float(row['mean_target_q']) - 15.220:+.3f})")
            print(f"    β2 · ratio              = 0.0111 (Δ priv − dep = {beta2 * penalty_ratio - 0.0111:+.4f})")
else:
    print(f"[warn] overview.csv missing at {overview_path}")

[overview] dataset=worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000 pair=actorb_4p0__criticb_2p0 num_seeds=3
  mean_test_success_rate = 0.9267
  std_test_success_rate  = 0.0249
  mean_test_return       = 11.990
  mean_critic_penalty    = 0.0807
  mean_target_q          = 15.349
  mean_critic_penalty_ratio = 0.0057  (β2 · ratio = 0.0113)

  对照（Phase 1 deployable 同 finalist）:
    mean_test_success_rate = 0.9280  (Δ priv − dep = -0.0013)
    mean_target_q           = 15.220 (Δ priv − dep = +0.129)
    β2 · ratio              = 0.0111 (Δ priv − dep = +0.0002)


## 5. seed 44 诊断（Phase 2 核心信息）

Phase 1 deployable 上 seed 44 的 test_succ 是 `0.78`（selected ckpt 卡在 ep40，val 仅 0.825）；其它 4 seed 都 ≥ 0.93。Phase 2 在 privileged-critic 轨道下重训 seed 44，直接区分两种假设：

- **假设 A**：privileged critic 把 seed 44 救回（test_succ ≥ 0.85）→ critic 侧的 OOD/信息不足是 seed 44 难训的真因；
- **假设 B**：privileged critic 救不回 seed 44（test_succ 仍 < 0.85）→ seed 44 是数据/优化层面 outlier，与 critic 无关；后续需要回到 collector 检查 seed 44 触发的 episode 起点分布。

In [ ]:
if not per_seed.empty:
    seed_44_row = per_seed[per_seed["seed"] == "44"]
    if seed_44_row.empty:
        print("[warn] seed 44 缺失")
    else:
        s44 = seed_44_row.iloc[0]
        s44_priv = float(s44["success_rate"])
        s44_dep  = 0.780  # Phase 1 实测
        delta    = s44_priv - s44_dep

        print("=" * 60)
        print("seed 44 跨轨道对比")
        print("-" * 60)
        print(f"  Phase 1 deployable:        success = {s44_dep:.3f}")
        print(f"  Phase 2 privileged-critic: success = {s44_priv:.3f}")
        print(f"  Δ (priv − dep):            {delta:+.3f} ({delta*100:+.1f}pp)")
        print("=" * 60)
        print()

        if s44_priv >= 0.85:
            verdict = "假设 A：privileged critic 把 seed 44 救回 → critic 信息瓶颈是真因"
        elif s44_priv >= 0.78 + 0.05:
            verdict = "中间档：privileged critic 部分救回 seed 44，需更多 seed 确认机制"
        else:
            verdict = "假设 B：privileged critic 救不回 seed 44 → 数据/优化层面 outlier，与 critic 无关"
        print(f"[seed 44 诊断结论] {verdict}")

seed 44 跨轨道对比
------------------------------------------------------------
  Phase 1 deployable:        success = 0.780
  Phase 2 privileged-critic: success = 0.900
  Δ (priv − dep):            +0.120 (+12.0pp)

[seed 44 诊断结论] 假设 A：privileged critic 把 seed 44 救回 → critic 信息瓶颈是真因


## 6. 与 Phase 1 / TD3BC worldcomp formal 四向对比

把 ReBRAC privileged-critic 加入 Stage D 主对照表。

In [ ]:
if not per_seed.empty:
    rebrac_priv_mean = per_seed["success_rate"].mean()
    rebrac_priv_std  = per_seed["success_rate"].std()
    rebrac_priv_ret  = per_seed["return"].mean()

    # Phase 1 deployable 实测
    rebrac_dep_mean  = 0.928
    rebrac_dep_std   = 0.077
    rebrac_dep_ret   = 20.17

    # TD3BC worldcomp formal
    td3bc_dep_mean   = 0.858
    td3bc_dep_std    = 0.080
    td3bc_priv_mean  = 0.922
    td3bc_priv_std   = 0.086

    teacher_mean     = 0.990

    print("=" * 84)
    print(f"{'Protocol':<54}{'mean':>8}{'std':>8}{'return':>10}")
    print("-" * 84)
    print(f"{'TD3BC worldcomp deployable (α=0.0, BC)':<54}{td3bc_dep_mean:>8.3f}{td3bc_dep_std:>8.3f}{-14.29:>10.2f}")
    print(f"{'TD3BC worldcomp privileged-critic (α=0.1)':<54}{td3bc_priv_mean:>8.3f}{td3bc_priv_std:>8.3f}{16.49:>10.2f}")
    print(f"{'ReBRAC worldcomp deployable Phase 1 (5 seeds)':<54}{rebrac_dep_mean:>8.3f}{rebrac_dep_std:>8.3f}{rebrac_dep_ret:>10.2f}")
    print(f"{'ReBRAC worldcomp privileged-critic Phase 2 (3 seeds)':<54}{rebrac_priv_mean:>8.4f}{rebrac_priv_std:>8.4f}{rebrac_priv_ret:>10.2f}")
    print(f"{'teacher baseline (online)':<54}{teacher_mean:>8.3f}{'-':>8}{32.19:>10.2f}")
    print("=" * 84)
    print()

    delta_td3bc_priv  = rebrac_priv_mean - td3bc_priv_mean
    delta_rebrac_dep  = rebrac_priv_mean - rebrac_dep_mean
    delta_teacher     = rebrac_priv_mean - teacher_mean

    print(f"Δ vs TD3BC privileged-critic = {delta_td3bc_priv:+.4f} ({delta_td3bc_priv*100:+.1f}pp)")
    print(f"Δ vs ReBRAC deployable Phase1= {delta_rebrac_dep:+.4f} ({delta_rebrac_dep*100:+.1f}pp)  ← 'privileged critic 是否在 ReBRAC 上仍贡献增益'")
    print(f"Δ vs teacher baseline        = {delta_teacher:+.4f} ({delta_teacher*100:+.1f}pp)")
    print()

    deployable_gap = teacher_mean - td3bc_dep_mean
    print(f"Gap closure (vs TD3BC deployable → teacher):")
    print(f"  TD3BC privileged-critic     = {(td3bc_priv_mean - td3bc_dep_mean) / deployable_gap * 100:5.1f}%")
    print(f"  ReBRAC deployable  Phase 1  = {(rebrac_dep_mean  - td3bc_dep_mean) / deployable_gap * 100:5.1f}%")
    print(f"  ReBRAC privileged  Phase 2  = {(rebrac_priv_mean - td3bc_dep_mean) / deployable_gap * 100:5.1f}%")

Protocol                                                  mean     std    return
------------------------------------------------------------------------------------
TD3BC worldcomp deployable (α=0.0, BC)                   0.858   0.080    -14.29
TD3BC worldcomp privileged-critic (α=0.1)                0.922   0.086     16.49
ReBRAC worldcomp deployable Phase 1 (5 seeds)            0.928   0.077     20.17
ReBRAC worldcomp privileged-critic Phase 2 (3 seeds)    0.9267  0.0306     11.99
teacher baseline (online)                                0.990       -     32.19

Δ vs TD3BC privileged-critic = +0.0047 (+0.5pp)
Δ vs ReBRAC deployable Phase1= -0.0013 (-0.1pp)  ← 'privileged critic 是否在 ReBRAC 上仍贡献增益'
Δ vs teacher baseline        = -0.0633 (-6.3pp)

Gap closure (vs TD3BC deployable → teacher):
  TD3BC privileged-critic     =  48.5%
  ReBRAC deployable  Phase 1  =  53.0%
  ReBRAC privileged  Phase 2  =  52.0%


## 7. Phase 2 通过判据核对

按 [plan §6.5.3](../docs/rebrac_experiment_plan.md)：

| 条件 | 阈值 | Phase 2 实测 | 是否通过 |
| --- | --- | --- | --- |
| Rule 1：mean test success | `> 0.922`（TD3BC privileged-critic formal） | （上 cell 计算） | |
| Rule 2：std test success | `≤ 0.10` | （上 cell 计算） | |
| Rule 3：gap closure（情景 A） | `> 50%`（vs TD3BC privileged 48.5%） | （上 cell 计算） | |
| Rule 4：seed 44 诊断 | `test_succ ≥ 0.85` ⇒ 假设 A | （cell 5 计算） | |

按这四条结果决定 Stage D 收口或追加：

1. **全部通过**：Stage D Phase 2 阳性结论成立，ReBRAC privileged-critic 进一步压缩 deployable→teacher gap，且 seed 44 故事在两轨道下统一。直接写入 `docs/rebrac_experiment_report.md` 新增小节，Stage D 收口。
2. **Rule 1/2 通过但 Rule 4 失败**（seed 44 在 privileged 仍 < 0.85）：写入报告；触发 collector 层面追查 seed 44 episode 起点分布；Stage E ablation 优先级仍维持。
3. **Rule 1 失败**（mean ≤ 0.922）：Phase 2 没有给出超越 TD3BC privileged 的额外收益；意味着 ReBRAC 的全部 worldcomp 增益已经在 Phase 1 deployable 取得（Finding 1 机制猜想成立）。这种情况下建议立即跑 critic-penalty-off probe（[notebooks/rebrac_worldcomp_critic_penalty_off_probe.ipynb](rebrac_worldcomp_critic_penalty_off_probe.ipynb)）确认 dual penalty 退化机制。

## 8. 报告写入清单

跑完上面所有 cell 后：

1. **更新 [docs/rebrac_experiment_report.md](../docs/rebrac_experiment_report.md)**：
   - 新增 §7.12 「Stage D Phase 2: `worldcomp-1000` privileged-critic formal」小节，含 per-seed 表、与 Phase 1 / TD3BC 四向对比、seed 44 诊断结论、Phase 2 通过判据核对；
   - §1 一句话当前状态更新；
   - §10 最终结论增加 Phase 2 outcome；
   - §11 文件索引追加 Phase 2 路径。
2. **更新 [docs/rebrac_experiment_plan.md](../docs/rebrac_experiment_plan.md)**：
   - §6.5.3 Phase 2 标记为 `【已完成】`；
   - §10 推荐执行顺序更新。
3. **决定 Stage E**：
   - 若 Rule 4 通过（privileged 救回 seed 44）→ Stage E 可保留 critic-penalty-off / normalize_q-off 两个最小 ablation；
   - 若 Rule 4 失败 → Stage E 优先做 collector 层面 seed 44 episode 起点分布检查，再决定 ablation 范围；
   - 若 Rule 1 失败 → Stage E 主任务变为 critic-penalty-off probe，验证 "ReBRAC 在 worldcomp 上的 dual penalty 退化为单 penalty" 机制猜想，作为 Stage E 收口结论。